In [24]:
import numpy as np
import random
import heapq
import re
from pprint import pprint
from math import ceil
from functools import total_ordering

In [25]:
TRACE_FILE="./skew_uniform_3.0_300.csv"
servers = [0,1,2,3]
backend="system"
step = 30

In [26]:
@total_ordering
class Request:
    def __init__(self, req_id, model_dir, adapter_dir, prompt, prompt_len, output_len, req_time):
        self.req_id = req_id
        self.model_dir = model_dir 
        self.adapter_dir = adapter_dir
        self.prompt = prompt
        self.prompt_len = prompt_len
        self.output_len = output_len
        self.req_time = req_time

    def __repr__(self):
        return f"req_id={self.req_id}, " \
               f"model_dir={self.model_dir}, adapter_dir={self.adapter_dir}, " \
               f"prompt_len={self.prompt_len}, output_len={self.output_len}, " \
               f"req_time={self.req_time}"

    def __eq__(self, other):
        return self.req_id == other.req_id

    def __lt__(self, other):
        return self.req_time < other.req_time

def dummy_prompt(prompt_len):
    return "Hello " * prompt_len


In [27]:
def read_requests(trace_file):
    requests = []
    adapter_dirs = set()
    with open(trace_file, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:
            elements = line.split(",")
            requests.append(
                Request(
                    req_id=int(elements[0]),
                    model_dir=elements[1],
                    adapter_dir=elements[2],
                    prompt=dummy_prompt(int(elements[3])),
                    prompt_len=int(elements[3]),
                    output_len=int(elements[4]),
                    req_time=float(elements[5]),
                )
            )
            # requests.append((int(elements[0]),elements[1],elements[2],int(elements[3]),int(elements[4]),float(elements[5])))
            adapter_dirs.add(elements[2])
    requests.sort(key=lambda r: r.req_time)
    return list(adapter_dirs), requests

In [28]:
adapter_dirs, requests = read_requests(trace_file=TRACE_FILE)
avg_prompt_len = np.mean([req.prompt_len for req in requests])
avg_output_len = np.mean([req.output_len for req in requests])
avg_len = np.mean([req.prompt_len + req.output_len for req in requests])
print(
    "num_adapters",
    len(adapter_dirs),
    "num_requests",
    len(requests),
    "avg_len:",
    avg_len,
    "avg_prompt_len:",
    avg_prompt_len,
    "avg_output_len:",
    avg_output_len,
)

num_adapters 25 num_requests 900 avg_len: 628.0 avg_prompt_len: 500.0 avg_output_len: 128.0


In [29]:
step_idx = 0
last_time = requests[0]
for req in requests:
    if req.req_time > last_time.req_time // 1 + step:
        demand_tps = {a: 0 for a in adapter_dirs}
        index = requests.index(last_time)
        while requests[index].req_time < req.req_time:
            # rank = int(re.search(r'rank-(\d+)', requests[index].adapter_dir).group(1))
            demand_tps[requests[index].adapter_dir] = (
                demand_tps.get(requests[index].adapter_dir)
                + (
                    requests[index].prompt_len
                    + requests[index].output_len
                )
                / step
            )
            index += 1
        adapter_demand = []
        for adapter, tps in demand_tps.items():
            rank = int(re.search(r"rank-(\d+)", adapter).group(1))
            adapter_demand.append((rank, tps, adapter))  # tps here is expected tps
        adapter_demand.sort(reverse=True)

        # server_tps = {8: 2400, 16: 2100, 32: 1900, 64:1700, 128:1600} # operating point, fn of max rank, old NC24ads-hipri 8xA100 40GB
        server_tps = {
            8: 2725,
            16: 2700,
            32: 2675,
            64: 2625,
            128: 2525,
        }  # operating point, fn of max rank, 4xA100 80GB

        adapter_groups = [[] for _ in servers]
        num_servers = len(servers)
        server_occupied_tps = [0] * num_servers
        server_max_rank = [0] * num_servers
        adapters_placed = [False] * len(adapter_demand)

        with open("allocation_log.txt", "a") as f:
            f.write("\n\n************************************")
            f.write(f"step {step_idx} @ time {last_time.req_time} to {req.req_time}:")

        def is_compatible(
            group_idx,
            tuple,
            server_max_rank=server_max_rank,
            server_occupied_tps=server_occupied_tps,
            scale=1,
        ):
            """
            Check if the given adapter tuple can fit within the group
            An adapter can fit if it is within the tps limit
            The tps limit depends on the max rank of the allocated adapters to this server
            """
            rank, tps, _ = tuple
            max_rank = max(rank, server_max_rank[group_idx])
            tps = server_occupied_tps[group_idx] + tuple[1]
            return tps <= (server_tps[max_rank] * scale)

        # * checking compatibility
        rank_instance_budget = [(rank, sum(tps for r, tps, _ in adapter_demand if r == rank) / rank_max_tps) for rank, rank_max_tps in server_tps.items()]
        sorted_budgets = sorted(rank_instance_budget, key=lambda x: x[1], reverse=True)
        assert sum(budget for _, budget in rank_instance_budget) <= num_servers, "Exceeded server budget"

        # * rounding
        rounded_budgets = [(rank, round(budget)) for rank, budget in sorted_budgets]
        sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)
        if sum_rounded_off_budgets < num_servers:
            idx = 0
            while sum_rounded_off_budgets < num_servers and idx < len(rounded_budgets):
                rounded_budgets[idx] = (rounded_budgets[idx][0], ceil(sorted_budgets[idx][1]))
                idx += 1
                sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)

        # * balanced allocation within assigned instances
        adapters_with_assigned_instances = [x for x in rounded_budgets if x[1] > 0]
        current_server = 0
        for rank, num_assigned_instances in adapters_with_assigned_instances:
            l = current_server
            r = current_server + num_assigned_instances
            server_heap = [(server_occupied_tps[i], i) for i in range(l, r)]
            heapq.heapify(server_heap)
            for adapter_idx, adapter in enumerate(adapter_demand):
                if adapter[0] == rank:
                    while server_heap:
                        occupancy, least_occupied_server = heapq.heappop(server_heap)
                        if is_compatible(least_occupied_server, adapter):
                            adapter_groups[least_occupied_server].append(adapter)
                            server_occupied_tps[least_occupied_server] += adapter[1]
                            adapters_placed[adapter_idx] = True
                            server_max_rank[least_occupied_server] = max(server_max_rank[least_occupied_server], rank)
                            heapq.heappush(server_heap, (server_occupied_tps[least_occupied_server], least_occupied_server))
                            break
                
            current_server += num_assigned_instances

        # * leftovers
        for adapter_idx, adapter in enumerate(adapter_demand):
            if not adapters_placed[adapter_idx]:
                least_occupied_server = min(
                    (i for i in range(num_servers) if server_max_rank[i] >= adapter[0]),
                    key=lambda x: server_occupied_tps[x],
                    default=None
                )
                if least_occupied_server is not None and is_compatible(least_occupied_server, adapter):
                    adapter_groups[least_occupied_server].append(adapter)
                    server_occupied_tps[least_occupied_server] += adapter[1]
                    adapters_placed[adapter_idx] = True
                    continue

                # we could not find a server with rank >= this adapters rank
                # need to colocate with a lower rank
                # TODO: better logic here - search through the closest ranks first and stop if we can fit
                new_least_occupied_server = min(range(num_servers), key=lambda x: server_occupied_tps[x])
                if is_compatible(new_least_occupied_server, adapter):
                    with open("allocation_log.txt", "a") as f:
                        f.write(
                            f"{last_time} Adapter {adapter[2]} with rank {adapter[0]}, tps {adapter[1]} could not be placed in a server with rank >= {adapter[0]}, placing in server {servers[new_least_occupied_server]} with max rank {server_max_rank[new_least_occupied_server]}\n"
                        )
                    adapter_groups[new_least_occupied_server].append(adapter)
                    server_occupied_tps[new_least_occupied_server] += adapter[1]
                    adapters_placed[adapter_idx] = True
                    server_max_rank[new_least_occupied_server] = max(server_max_rank[new_least_occupied_server], adapter[0])


        with open("allocation_log.txt", "a") as f:
            f.write(f"\n{last_time} Adapter groups:\n")
            for i, group in enumerate(adapter_groups):
                f.write(
                    f"  Server {servers[i]}: {[(adapter, tps) for _, tps, adapter in group]}\n"
                )
                f.write(
                    f"  Server {servers[i]} total tps: {server_occupied_tps[i]}\n"
                )
                f.write(
                    f"  Server {servers[i]} max tps: {[server_tps[server_max_rank[i]] for i in range(num_servers)]}\n"
                )
                f.write(
                    f"  Server {servers[i]} max rank: {server_max_rank[i]}\n"
                )
            f.write("************************************\n\n")

        server_map = {}
        for i, server in enumerate(servers):
            for _, _, adapter in adapter_groups[i]:
                server_map[adapter] = server
        step_idx += 1
        last_time = req

In [30]:
def reset_allocation_log():
    with open("allocation_log.txt", "w") as f:
        f.write("")